
# Define the role of AI and identify your candidates


AI will recognize human-object interations in video, such as a person picking up a backpack, sitting down at a desk, etc. The input will be frames of live-video. The system will use YOLO to identify the object and its bounding boxes which will then be processed in MediaPipe to estimate body pose, and temporal actions.

Non-AI comonent consists of video capture using openCV, calculating distances from human to the object, etc. These actions will have a list of best actions that can be suggested by the system.

# Load and inspect your data
The dataset for this project is a video, only proprocessing required was trimming, sizing, and editing the video to be standardize. 

# Run your first candidate in its simplest working form


For this milestone, I compared performance between several YOLO and pose_landmarks model. Between yolov8n, yolov8s, and yolov8l, 8l model performed significantly better than the others. However, givent hat the model was larger, its processing time was also significantly larger. I chose to settle for yoloyv8s, a good balance between accuracy and efficiency for this milestone. 

The outputs were not entirely accurate but good enough for this stage. Some objects were not correctly identified or not identified at all. Some actions were not being identified (drinking out of a cup or picking up objects). 

However, the system identified most objects and actions in a controlled environment (person intentionally moving slowly, placing objects close and still to the camera). Basic actions such as reaching for a certain object, interacting with an object, and touching was correctly identified

# Evaluate the initial results
Choose at least one metric that reflects your system's purpose and explain why it is appropriate. Evaluate your first configuration on validation data or designated development evaluation cases. Record the score and number of examples evaluated. Use the table below for ideas, and add a visualization if it helps explain your results. Treat results from a small sample or short training run as preliminary.



# Inspect errors and practical constraints

1. Not identifying action: picking up an item
2. Not registering an event in quick motion
3. Objects are not correctly identified. 

Practical Constraints:
1. Latency; with basic models with limited hardware capabilities on local machine, the processing time is long for a 12 seconds video (~5 min). 
2. API cost will certainly be an issue when using a more powerful model in production
3. TBD

# Make one targeted change and run again
Use your results or error analysis to choose one change. For example, adjust a learning rate, training budget, decision threshold, prompt, retrieval setting, or agent instruction. State what you expect the change to improve and why. Keep other settings fixed where practical so you can interpret the effect.

Run the revised configuration and evaluate it on the same development examples with the same metric. Record both configurations and their results, then explain whether the change helped, hurt, or produced no clear improvement. You do not need extensive tuning or a model search for this homework. If a technical blocker prevents the second run, document the error, what you tried, and the next action needed.

A large part of inaccuracy was due to weak pre-trained models. Using powerful models such as yolov8l and pose_landmark_heavy was able to correctly identify all objects as well as actions. 

I am planning on tweaking the decision thresholds for actions see below ```constant tunables``` section for the said thresholds

# imports

In [16]:
import time, math, collections
import cv2, numpy as np, mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from ultralytics import YOLO

# Constants such as tunable settings, body coordinates, object ID, etc.

In [10]:
# ---------- what the system knows about (COCO class ids) ----------
CLASS_NAMES = {24: "backpack", 26: "handbag", 28: "suitcase", 39: "bottle",
               41: "cup", 63: "laptop", 67: "cell phone", 73: "book",
               56: "chair", 57: "couch"}
PORTABLE = {"backpack", "handbag", "suitcase", "bottle", "cup",
            "cell phone", "book", "laptop"}
SEATS    = {"chair", "couch"}

FACE_USE = {"cup": "drinking from", "bottle": "drinking from", "cell phone": "using"}

SUGGESTIONS = {
    "picking up":    "Log pickup event",
    "holding":       "Object in use / being carried",
    "putting down":  "Log object released",
    "reaching for":  "Person about to interact",
    "touching":      "Contact with object",
    "drinking from": "Hydration detected",
    "using":         "Device in use",
    "sitting":       "Person seated",
    "":              "",
}

# ---------- tunable settings ----------
YOLO_CONF    = 0.3
NEAR_FRAC    = 0.35   # "near" = wrist within 25% of box's larger side
RISE_FRAC    = 0.04   # object must rise 5% of frame height while touched
RELEASE_GAP  = 5     # frames without a nearby wrist before "released"
PICK_SHOW    = 15     # frames to keep showing "picking up"
PUT_SHOW     = 15    # frames to keep showing "putting down"
REACH_FRAMES = 3     # near for fewer frames than this = "reaching"
FACE_FRAC    = 0.20   # object within 12% of frame height of mouth
SIT_KNEE_MARGIN = 0.10
SMOOTH       = 3
WRISTS, HIPS, KNEES, MOUTH = (15, 16), (23, 24), (25, 26), (9, 10)

POSE_CONNECTION = [
    (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8), (9, 10),
    (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
]

# Helper functions

In [ ]:

class ObjState:
    def __init__(self):
        self.base = None; self.held = False; self.gap = 0
        self.near_count = 0; self.pick_until = -1; self.put_until = -1
        self.last_seen = 0

def point_box_dist(px, py, box):
    x1, y1, x2, y2 = box
    return math.hypot(max(x1 - px, 0, px - x2), max(y1 - py, 0, py - y2))

def vis_ok(lm, t=0.4):
    return getattr(lm, "visibility", 1.0) > t

def make_landmarker():
    return vision.PoseLandmarker.create_from_options(
        vision.PoseLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path="POSES/pose_landmarker_full.task"),
            running_mode=vision.RunningMode.VIDEO,
            num_poses=1,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5,
        )
    )

# With Camera

In [ ]:


# def main(camera=1):
#     pose_landmarker = make_landmarker()
#     yolo = YOLO("yolov8l.pt")
#     cap = cv2.VideoCapture(camera)
#     if not cap.isOpened():
#         pose_landmarker.close()
#         raise RuntimeError("Cannot open webcam")

#     states = {}
#     votes = collections.deque(maxlen=SMOOTH)
#     start = time.monotonic(); last_ts = -1; fi = 0

#     try:
#         while True:
#             ok, frame = cap.read()
#             if not ok: break
#             frame = cv2.flip(frame, 1)
#             h, w = frame.shape[:2]
#             rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#             fi += 1

#             ts = max(int((time.monotonic() - start) * 1000), last_ts + 1); last_ts = ts
#             res = pose_landmarker.detect_for_video(
#                 mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts)

#             # ---- 1. YOLO detection + tracking ----
#             tr = yolo.track(frame, persist=True, conf=YOLO_CONF,
#                             classes=list(CLASS_NAMES), verbose=False)[0]
#             objs = []                                   # (id, name, box)
#             if tr.boxes is not None and tr.boxes.id is not None:
#                 for b, tid in zip(tr.boxes, tr.boxes.id.int().tolist()):
#                     objs.append((tid, CLASS_NAMES[int(b.cls[0])], b.xyxy[0].tolist()))

#             # ---- 2. Pose ----
#             wrists, mouth, hip_pt, knee_y = [], None, None, None
#             if res.pose_landmarks:
#                 pose = res.pose_landmarks[0]
#                 for a, b in POSE_CONNECTION:
#                     cv2.line(frame, (int(pose[a].x*w), int(pose[a].y*h)),
#                                     (int(pose[b].x*w), int(pose[b].y*h)), (255, 255, 0), 2)

                
#                 for lm in pose:
#                     cv2.circle(frame, (int(lm.x*w), int(lm.y*h)), 3, (0, 255, 0), -1)
#                 wrists = [(pose[i].x*w, pose[i].y*h) for i in WRISTS if vis_ok(pose[i])]
#                 mouth = (np.mean([pose[i].x for i in MOUTH])*w,
#                          np.mean([pose[i].y for i in MOUTH])*h)
#                 if all(vis_ok(pose[i]) for i in HIPS):
#                     hip_pt = (np.mean([pose[i].x for i in HIPS])*w,
#                               np.mean([pose[i].y for i in HIPS])*h)
#                 if all(vis_ok(pose[i]) for i in KNEES):
#                     knee_y = np.mean([pose[i].y for i in KNEES]) * h

#             # ---- 3. Sitting ----
#             sitting = False
#             if hip_pt:
#                 for _, name, (x1, y1, x2, y2) in objs:
#                     if name in SEATS and x1 < hip_pt[0] < x2 and y1 < hip_pt[1] < y2:
#                         if knee_y is None or abs(knee_y - hip_pt[1]) < SIT_KNEE_MARGIN * h:
#                             sitting = True

#             # ---- 4. Per-object interaction state machine ----
#             candidates = []                             # (priority, verb, text)
#             for tid, name, box in objs:
#                 x1, y1, x2, y2 = box
#                 color = (0, 165, 255) if name in PORTABLE else (200, 200, 200)
#                 cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
#                 cv2.putText(frame, f"{name} #{tid}", (int(x1), int(y1) - 5),
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
#                 if name not in PORTABLE: continue

#                 st = states.setdefault(tid, ObjState()); st.last_seen = fi
#                 cy = (y1 + y2) / 2
#                 near_px = NEAR_FRAC * max(x2 - x1, y2 - y1)
#                 near = bool(wrists) and min(point_box_dist(x, y, box) for x, y in wrists) < near_px

#                 if near:
#                     st.gap = 0; st.near_count += 1
#                     if st.base is None: 
#                         st.base = cy
                        
#                     if not st.held and (st.base - cy) > RISE_FRAC * h:
#                         st.held = True; st.pick_until = fi + PICK_SHOW
#                 else:
#                     st.gap += 1; st.near_count = 0
#                     if st.gap > RELEASE_GAP:
#                         if st.held: st.put_until = fi + PUT_SHOW
#                         st.held = False; st.base = None

#                 if st.held and fi < st.pick_until:
#                     candidates.append((6, "picking up", f"picking up {name}"))
#                 elif st.held:
#                     near_face = mouth and point_box_dist(*mouth, box) < FACE_FRAC * h
#                     if near_face and name in FACE_USE:
#                         v = FACE_USE[name]; candidates.append((4, v, f"{v} {name}"))
#                     else:
#                         candidates.append((3, "holding", f"holding {name}"))
#                 elif fi < st.put_until:
#                     candidates.append((5, "putting down", f"putting down {name}"))
#                 elif near and name == "laptop" and sitting:
#                     candidates.append((4, "using", "using laptop"))
#                 elif near:
#                     v = "reaching for" if st.near_count < REACH_FRAMES else "touching"
#                     candidates.append((2, v, f"{v} {name}"))

#             if sitting:
#                 candidates.append((1, "sitting", "sitting"))

#             for t in [t for t, s in states.items() if fi - s.last_seen > 60]:
#                 del states[t]                           # forget objects gone ~2 s

#             # ---- 5. Smooth + display ----
#             best = max(candidates, key=lambda c: c[0]) if candidates else (0, "", "none")
#             votes.append(best[1:])
#             (verb, text), _ = collections.Counter(votes).most_common(1)[0]

#             cv2.rectangle(frame, (0, 0), (w, 60), (0, 0, 0), -1)
#             cv2.putText(frame, f"Action: {text}", (10, 28),
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
#             cv2.putText(frame, SUGGESTIONS.get(verb, ""), (10, 52),
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
            

#             cv2.imshow("Action detector", frame)
#             if cv2.waitKey(1) & 0xFF == 27:             # Esc to quit
#                 break
#     finally:
#         cap.release()
#         pose_landmarker.close()
#         cv2.destroyAllWindows()
#         cv2.waitKey(1)

# main()

# Integrate with Qwen3:4b, TBC

In [28]:
import json, re, ollama, threading

LLM_MODEL = "qwen3:4b"
ALLOWED_ACTIONS = ["picking up", "holding", "putting down", "reaching for", "touching",
                   "drinking from", "using", "sitting", "standing", "walking away", "none"]
ALLOWED_SUGGESTIONS = list(SUGGESTIONS.values()) + ["No action needed"]


PROMPT = f"""You interpret a stream of events from a computer-vision system that tracks
a person and nearby objects. Events come from object detection and pose geometry,
so they can be noisy: labels may flicker, and detections can drop out briefly.

Given the recent events, return the most reasonable action the person is doing. For example, if a person is picking up a backpack, 
it is very likely the person is going to study or is going to work. 

Output should be in the format of suggested action and reason for example, "person is picking up a backpack, good chance they are going to school"

Rules: use only the evidence in the events, treat very short events (<0.3 s) as noise,
and use low confidence if the events conflict. Do not invent objects that are not listed."""

# With pre-recorded video

In [26]:
def main(video_path, save_path=None, mirror=False, display_width=960):
    pose_landmarker = make_landmarker()
    yolo = YOLO("YOLO/yolov8s.pt")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        pose_landmarker.close()
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"{video_path}: {W}x{H}, {fps:.1f} fps, {total} frames")

    writer = None
    if save_path:
        writer = cv2.VideoWriter(save_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    states = {}
    votes = collections.deque(maxlen=SMOOTH)
    last_ts = -1; fi = 0

    # Keep track of actions and objects
    records = []          # one dict per frame
    events = []           # one dict per action segment (label changes)
    cur = None           

    try:
        while True:
            ok, frame = cap.read()
            if not ok: break                            # end of video
            if mirror: frame = cv2.flip(frame, 1)       # off by default for recorded video
            h, w = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            fi += 1

            # timestamp comes from the video's frame number, not the wall clock
            ts = max(int((fi - 1) * 1000 / fps), last_ts + 1); last_ts = ts
            res = pose_landmarker.detect_for_video(
                mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts)

            # ---- 1. YOLO detection + tracking ----
            tr = yolo.track(frame, persist=True, conf=YOLO_CONF,
                            classes=list(CLASS_NAMES), verbose=False)[0]
            objs = []                                   # (id, name, box)
            if tr.boxes is not None and tr.boxes.id is not None:
                for b, tid in zip(tr.boxes, tr.boxes.id.int().tolist()):
                    objs.append((tid, CLASS_NAMES[int(b.cls[0])], b.xyxy[0].tolist()))

            # ---- 2. Pose ----
            wrists, mouth, hip_pt, knee_y = [], None, None, None
            if res.pose_landmarks:
                pose = res.pose_landmarks[0]
                for a, b in POSE_CONNECTION:
                    cv2.line(frame, (int(pose[a].x*w), int(pose[a].y*h)),
                                    (int(pose[b].x*w), int(pose[b].y*h)), (255, 255, 0), 2)
                for lm in pose:
                    cv2.circle(frame, (int(lm.x*w), int(lm.y*h)), 3, (0, 255, 0), -1)
                wrists = [(pose[i].x*w, pose[i].y*h) for i in WRISTS if vis_ok(pose[i])]
                mouth = (np.mean([pose[i].x for i in MOUTH])*w,
                         np.mean([pose[i].y for i in MOUTH])*h)
                if all(vis_ok(pose[i]) for i in HIPS):
                    hip_pt = (np.mean([pose[i].x for i in HIPS])*w,
                              np.mean([pose[i].y for i in HIPS])*h)
                if all(vis_ok(pose[i]) for i in KNEES):
                    knee_y = np.mean([pose[i].y for i in KNEES]) * h

            # ---- 3. Sitting ----
            sitting = False
            if hip_pt:
                for _, name, (x1, y1, x2, y2) in objs:
                    if name in SEATS and x1 < hip_pt[0] < x2 and y1 < hip_pt[1] < y2:
                        if knee_y is None or abs(knee_y - hip_pt[1]) < SIT_KNEE_MARGIN * h:
                            sitting = True

            # ---- 4. Per-object interaction state machine ----
            candidates = []                             # (priority, verb, text)
            for tid, name, box in objs:
                x1, y1, x2, y2 = box
                color = (0, 165, 255) if name in PORTABLE else (200, 200, 200)
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                cv2.putText(frame, f"{name} #{tid}", (int(x1), int(y1) - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
                if name not in PORTABLE: continue

                st = states.setdefault(tid, ObjState()); st.last_seen = fi
                cy = (y1 + y2) / 2
                near_px = NEAR_FRAC * max(x2 - x1, y2 - y1)
                near = bool(wrists) and min(point_box_dist(x, y, box) for x, y in wrists) < near_px

                if near:
                    st.gap = 0; st.near_count += 1
                    if st.base is None:
                        st.base = cy
                    if not st.held and (st.base - cy) > RISE_FRAC * h:
                        st.held = True; st.pick_until = fi + PICK_SHOW
                else:
                    st.gap += 1; st.near_count = 0
                    if st.gap > RELEASE_GAP:
                        if st.held: st.put_until = fi + PUT_SHOW
                        st.held = False; st.base = None

                if st.held and fi < st.pick_until:
                    candidates.append((6, "picking up", f"picking up {name}"))
                elif st.held:
                    near_face = mouth and point_box_dist(*mouth, box) < FACE_FRAC * h
                    if near_face and name in FACE_USE:
                        v = FACE_USE[name]; candidates.append((4, v, f"{v} {name}"))
                    else:
                        candidates.append((3, "holding", f"holding {name}"))
                elif fi < st.put_until:
                    candidates.append((5, "putting down", f"putting down {name}"))
                elif near and name == "laptop" and sitting:
                    candidates.append((4, "using", "using laptop"))
                elif near:
                    v = "reaching for" if st.near_count < REACH_FRAMES else "touching"
                    candidates.append((2, v, f"{v} {name}"))

            if sitting:
                candidates.append((1, "sitting", "sitting"))

            for t in [t for t, s in states.items() if fi - s.last_seen > 60]:
                del states[t]                           # forget objects gone ~2 s

            # ---- 5. Smooth + display ----
            best = max(candidates, key=lambda c: c[0]) if candidates else (0, "", "none")
            votes.append(best[1:])
            (verb, text), _ = collections.Counter(votes).most_common(1)[0]

            t_now = round((fi - 1) / fps, 3)

            # which object is involved in the winning action (if any)
            obj_name = None
            for tid, name, box in objs:
                if name in text:
                    obj_name = name
                    break

            records.append(dict(
                frame=fi,
                time_s=t_now,
                action=verb or "none",              # e.g. "picking up"
                label=text,                         # e.g. "picking up cup"
                object=obj_name,                    # e.g. "cup"
                raw_label=best[2],                  # unsmoothed, for debugging
                objects_visible=[(tid, n) for tid, n, _ in objs],
                n_wrists=len(wrists),
                sitting=sitting,
            ))

            # collapse frames into events: one row per run of the same label
            if cur is None or cur[0] != text:
                if cur is not None:
                    events.append(dict(label=cur[0], start_s=cur[1],
                                       end_s=t_now, dur_s=round(t_now - cur[1], 2)))
                cur = (text, t_now)

            cv2.rectangle(frame, (0, 0), (w, 60), (0, 0, 0), -1)
            cv2.putText(frame, f"Action: {text}", (10, 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
            cv2.putText(frame, SUGGESTIONS.get(verb, ""), (10, 52),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
            cv2.putText(frame, f"frame {fi}/{total}", (w - 170, 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

            if writer: writer.write(frame)

            disp = frame
            if display_width and w > display_width:
                disp = cv2.resize(frame, (display_width, int(h * display_width / w)))
            cv2.imshow("Action detector", disp)
            if cv2.waitKey(1) & 0xFF == 27:             # Esc to quit early
                break
    finally:
        
        cap.release()
        if writer: writer.release()
        pose_landmarker.close()
        cv2.destroyAllWindows()
        cv2.waitKey(1)

    if cur is not None:
        events.append(dict(label=cur[0], start_s=cur[1],
                           end_s=round((fi - 1) / fps, 3),
                           dur_s=round((fi - 1) / fps - cur[1], 2)))
    return records, events

main("resources/going_home.mov", save_path="output_annotated.mp4")

resources/going_home.mov: 1080x1920, 60.0 fps, 743 frames


([{'frame': 1,
   'time_s': 0.0,
   'action': 'reaching for',
   'label': 'reaching for laptop',
   'object': 'laptop',
   'raw_label': 'reaching for laptop',
   'objects_visible': [(1, 'laptop'), (2, 'chair'), (3, 'cup')],
   'n_wrists': 1,
   'sitting': False},
  {'frame': 2,
   'time_s': 0.017,
   'action': 'reaching for',
   'label': 'reaching for laptop',
   'object': 'laptop',
   'raw_label': 'reaching for laptop',
   'objects_visible': [(1, 'laptop'), (2, 'chair'), (3, 'cup')],
   'n_wrists': 1,
   'sitting': False},
  {'frame': 3,
   'time_s': 0.033,
   'action': 'reaching for',
   'label': 'reaching for laptop',
   'object': 'laptop',
   'raw_label': 'touching laptop',
   'objects_visible': [(1, 'laptop'), (2, 'chair'), (3, 'cup')],
   'n_wrists': 1,
   'sitting': False},
  {'frame': 4,
   'time_s': 0.05,
   'action': 'reaching for',
   'label': 'reaching for laptop',
   'object': 'laptop',
   'raw_label': 'using laptop',
   'objects_visible': [(1, 'laptop'), (2, 'chair'), (3,

In [27]:
records, events = main("resources/studying.mov", save_path="output_annotated.mp4")

import pandas as pd
df = pd.DataFrame(records)
ev = pd.DataFrame(events)

print(ev)                                    # timeline of actions
print(df.action.value_counts())              # how often each action occurred
print(df[df.action == "picking up"])         # every frame of a specific action
df.to_csv("frames.csv", index=False)
ev.to_csv("events.csv", index=False)

resources/studying.mov: 1080x1920, 58.6 fps, 827 frames
                label  start_s   end_s  dur_s
0             sitting    0.000   2.066   2.07
1                none    2.066   2.340   0.27
2             sitting    2.340   2.784   0.44
3                none    2.784   2.801   0.02
4             sitting    2.801   3.108   0.31
5                none    3.108   3.262   0.15
6             sitting    3.262   4.355   1.09
7                none    4.355   4.953   0.60
8             sitting    4.953   6.285   1.33
9                none    6.285   6.319   0.03
10            sitting    6.319   6.353   0.03
11               none    6.353   6.490   0.14
12  reaching for book    6.490   6.524   0.03
13      touching book    6.524   6.797   0.27
14               none    6.797   6.951   0.15
15            sitting    6.951   7.907   0.96
16       using laptop    7.907   8.061   0.15
17            sitting    8.061   8.214   0.15
18       using laptop    8.214   8.658   0.44
19            sitting   